In [1]:
import os
import glob
import numpy as np
import pandas as pd
import pickle

# import Python3_OpenOE_AC_map_functions_v1_08_30s as oem
import mz_LFP_functions as mz_LFP

%matplotlib inline
%load_ext autoreload
%autoreload 2

---

# Load some necessary variables

In [2]:
insert_depth = 3100  #change this as appropriate

sp_bw_ch = 20/2

surface_ch = np.round(insert_depth/sp_bw_ch)
V1_hip_ch = np.round((insert_depth-1000)/sp_bw_ch)
Hip_thal_ch = np.round((insert_depth-1000-1200)/sp_bw_ch)

CA1_DG_ch = np.round((insert_depth-1000-600)/sp_bw_ch)

print(surface_ch, V1_hip_ch, Hip_thal_ch, CA1_DG_ch)

310.0 210.0 90.0 150.0


In [3]:
samples_tr = 7350 #this is based on the shortest #samples in a trial
sr = 2500
n_chan = 384
rec_length = 3.0 #how long is the arduino triggered?

---

# 1SEC DELAY RECORDINGS (novel at bottom of script)

In [4]:
start_path_ls=glob.glob(r"G:/Neuropixels/interval_operant_training/operant/raw_data/"+'oper*')
total_rec = 156

end_path = r"\continuous\Neuropix-PXI-100.1\continuous.dat"

In [5]:
alldatals=[]
raw_CC_ls = []
for start_path in start_path_ls:
    ls = []
    CC_num = start_path.split('\\')[-1].split('_')[1]
    et = start_path.split('\\')[-1].split('_')[1] + "_" + start_path.split('\\')[-1].split('_')[2]
    
    for i in range(6, total_rec): # remember, you have to exclude the first short TTL signal trials
        exp_rec_path = rf"\experiment1\recording{i}"
        fileName = start_path + exp_rec_path + end_path
        data = np.memmap(fileName, dtype='int16', mode='c')
        data2 = data.reshape(-1, n_chan)
        ls.append(data2[:samples_tr, 0:384])
        
    alldatals.append(ls)
    raw_CC_ls.append(CC_num)

# alldatals #18 x 150 x 7350 x 384 or rather num_mice x num_trials x num_samples x num_ch
print(len(raw_CC_ls))


18


# Split the recordings by case

Ignore these two cells if you're just looking at the novel data

In [6]:
#Psuedo random presentation of the stimuli - 25 per row * 6 rows = 150 trials
# 0 -- drifting grating, rewarded stimulus, 100 trials
# 1 -- pink noise, unrewarded stimulus, 50 trials
stim_order = [0,1,1,0,0,1,0,0,1,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,
               0,1,0,1,0,1,0,0,1,1,0,0,0,1,0,1,0,0,1,1,0,0,0,0,1,
               0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,1,0,0,0,0,1,
               0,0,0,1,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,
               1,0,0,0,1,0,1,0,0,0,1,0,1,0,0,1,0,0,0,1,0,1,1,0,0,
               0,0,1,0,0,1,0,0,1,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0]

#Psuedo random distribution of water to the rewarded stimuli
# 0 -- water given -- 80 times
# 1 -- no water given -- 20 times
rew_order = [0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,
               0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,
               0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,
               0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0]

overall_order = []
i=0
for idx,val in enumerate(stim_order):
    if val == 0:
        i = i+1
        if rew_order[i-1] == 0:
            overall_order.append('0') # rewH20
        elif rew_order[i-1] == 1:
            overall_order.append('1') # rewnoH20
    elif val == 1:
        overall_order.append('2')     # unrew
print(len(overall_order))

150


In [7]:
rew_ls = [[] for i in range(len(alldatals))]
rew2_ls = [[] for i in range(len(alldatals))]
unrew_ls = [[] for i in range(len(alldatals))]


for ii in range(len(alldatals)):
    for idx,val in enumerate(overall_order):
        if val == '0':
            rew_ls[ii].append(alldatals[ii][idx])
        elif val == '1':
            rew2_ls[ii].append(alldatals[ii][idx])
        elif val == '2':
            unrew_ls[ii].append(alldatals[ii][idx])
        
        
print('rewarded trials with h2o: {0}'.format(len(rew_ls[0])))
print('rewarded trials without h2o: {0}'.format(len(rew2_ls[0])))
print('unrewarded trials: {0}'.format(len(unrew_ls[0])))

rewarded trials with h2o: 80
rewarded trials without h2o: 20
unrewarded trials: 50


# Load previous saved arrays to find strongest channel

In [8]:
all_rew_arr = np.load(r"D:\mz_Data\saved_dfs\Operant_reward\reward\lfp_npy\1sec_all_rew.npy")
all_rew2_arr = np.load(r"D:\mz_Data\saved_dfs\Operant_reward\reward\lfp_npy\1sec_all_rew2.npy")
all_unrew_arr = np.load(r"D:\mz_Data\saved_dfs\Operant_reward\reward\lfp_npy\1sec_all_unrew.npy")

In [9]:
pkl_file = r"D:\mz_Data\saved_dfs\Operant_reward\reward\lfp_npy\1sec_cc_ls"

open_file = open(pkl_file, "rb")
CC_ls = pickle.load(open_file)
open_file.close()


In [10]:
print('rew: {0}'.format(all_rew_arr.shape))
print('rew2: {0}'.format(all_rew2_arr.shape))
print('unrew: {0}'.format(all_unrew_arr.shape))

rew: (18, 384, 7350)
rew2: (18, 384, 7350)
unrew: (18, 384, 7350)


In [11]:
def WTFX_groups(data_array):
    groupA = []
    groupB = []
    for i in range(data_array.shape[0]):
        if (CC_ls[i] == "CC082263") | (CC_ls[i] == "CC067489") | (CC_ls[i] == "CC082260") | (CC_ls[i] == "CC084621"):
            groupA.append(data_array[i])
        elif (CC_ls[i] == "CC082257") | (CC_ls[i] == "CC067431") | (CC_ls[i] == "CC067432") | (CC_ls[i] == "CC082255"):
            groupB.append(data_array[i])
    groupA_arr = np.array(groupA)
    groupB_arr = np.array(groupB)
    return groupA_arr, groupB_arr

def find_strongest_ch(group_arr):
    mean_arr = group_arr.mean(axis=0)
    V1 = mean_arr[200:300, :]
    minim = np.where(V1 == np.amin(V1))
    min_ch = minim[0][0] + 199 # because the first channel is "0" not "1"
    print(min_ch)
    return min_ch

In [12]:
# Rewarded with water
wt_rew, fx_rew = WTFX_groups(all_rew_arr)

wt_rew_ch = find_strongest_ch(wt_rew)
fx_rew_ch = find_strongest_ch(fx_rew)

264
272


In [13]:
# Rewarded without water
wt_rew2, fx_rew2 = WTFX_groups(all_rew2_arr)

wt_rew2_ch = find_strongest_ch(wt_rew2)
fx_rew2_ch = find_strongest_ch(fx_rew2)

264
268


In [14]:
# Unrewarded
wt_unrew, fx_unrew = WTFX_groups(all_unrew_arr)

wt_unrew_ch = find_strongest_ch(wt_unrew)
fx_unrew_ch = find_strongest_ch(fx_unrew)

264
272


---

# Transform the lists into CAR filtered arrays
Selecting the activity from the channel with the strongest response given above

### First, separate each situation list into the WT/FX groups

In [15]:
def raw_WTFX_groups(data_ls):
    groupA = []
    groupB = []
    for i in range(len(data_ls)):
        if (raw_CC_ls[i] == "CC082263") | (raw_CC_ls[i] == "CC067489") | (raw_CC_ls[i] == "CC082260") | (raw_CC_ls[i] == "CC084621"):
            groupA.append(data_ls[i])
        elif (raw_CC_ls[i] == "CC082257") | (raw_CC_ls[i] == "CC067431") | (raw_CC_ls[i] == "CC067432") | (raw_CC_ls[i] == "CC082255"):
            groupB.append(data_ls[i])
    groupA_arr = np.array(groupA)
    groupB_arr = np.array(groupB)
    return groupA_arr, groupB_arr

In [16]:
raw_wt_rew, raw_fx_rew = raw_WTFX_groups(rew_ls)
raw_wt_rew2, raw_fx_rew2 = raw_WTFX_groups(rew2_ls)
raw_wt_unrew, raw_fx_unrew = raw_WTFX_groups(unrew_ls)

### Second, apply CAR and get correct output structure for PAC analysis
It should be in the shape of --- n trials x n samples ---

In [17]:
times = np.linspace(0, samples_tr/sr, samples_tr)
scale_factor = 0.195
time_arr = [0,0.5,1,1.5,2,2.5]
time_plot = [i*sr for i in time_arr]

In [18]:
# this function creates final CAR filtered arrays for rew, rew2, & unrew for each WT/FX group
# uses "applyCAR" and "notch_filt" functions

def create_reward_arr(reward_list, min_channel):
    rew_ls=[]
    for ii in range(len(reward_list)): #loop through each mouse
        sing_mouse=[]
        for ix in range(len(reward_list[0])): #loop through each trial
            tmp2 = reward_list[ii][ix]
            tmp2 = tmp2.T

            filt_tmp2 = []
            for ch in range(tmp2.shape[0]):
                ch_data = tmp2[ch,:]
                ch_notc_data = mz_LFP.notch_filt(ch_data)
                filt_tmp2.append(ch_notc_data)
            filt_tmp2 = np.array(filt_tmp2)

            CARfilt_tmp2 = mz_LFP.applyCAR(filt_tmp2, pr=0)

            scaled_CARfilt_tmp2 = CARfilt_tmp2[min_channel]*scale_factor

            sing_mouse.append(scaled_CARfilt_tmp2)

        rew_ls.append(sing_mouse)
        print('Loaded mouse {0}'.format(ii))
    rew_arr = np.array(rew_ls)
    
    return rew_arr

In [19]:
# For HPC pac analysis, the channel I'm using is 110
wt_rew_ch = 110
fx_rew_ch = 110
wt_rew2_ch = 110
fx_rew2_ch = 110
wt_unrew_ch = 110
fx_unrew_ch = 110

In [20]:
#Rewarded trials
rew_pac_wt = create_reward_arr(raw_wt_rew, wt_rew_ch)
rew_pac_fx = create_reward_arr(raw_fx_rew, fx_rew_ch)
print('Done with rew groups')
#Rewarded2 trials
rew2_pac_wt = create_reward_arr(raw_wt_rew2, wt_rew2_ch)
rew2_pac_fx = create_reward_arr(raw_fx_rew2, fx_rew2_ch)
print('Done with rew2 groups')
# Unrewarded trials
unrew_pac_wt = create_reward_arr(raw_wt_unrew, wt_unrew_ch)
unrew_pac_fx = create_reward_arr(raw_fx_unrew, fx_unrew_ch)
print('Done with unrew groups')
print('DONE WITH ALL GROUPS AND SITUATIONS -- Ready for PAC analysis')

Loaded mouse 0
Loaded mouse 1
Loaded mouse 2
Loaded mouse 3
Loaded mouse 4
Loaded mouse 5
Loaded mouse 6
Loaded mouse 7
Loaded mouse 8
Loaded mouse 9
Loaded mouse 0
Loaded mouse 1
Loaded mouse 2
Loaded mouse 3
Loaded mouse 4
Loaded mouse 5
Loaded mouse 6
Loaded mouse 7
Done with rew groups
Loaded mouse 0
Loaded mouse 1
Loaded mouse 2
Loaded mouse 3
Loaded mouse 4
Loaded mouse 5
Loaded mouse 6
Loaded mouse 7
Loaded mouse 8
Loaded mouse 9
Loaded mouse 0
Loaded mouse 1
Loaded mouse 2
Loaded mouse 3
Loaded mouse 4
Loaded mouse 5
Loaded mouse 6
Loaded mouse 7
Done with rew2 groups
Loaded mouse 0
Loaded mouse 1
Loaded mouse 2
Loaded mouse 3
Loaded mouse 4
Loaded mouse 5
Loaded mouse 6
Loaded mouse 7
Loaded mouse 8
Loaded mouse 9
Loaded mouse 0
Loaded mouse 1
Loaded mouse 2
Loaded mouse 3
Loaded mouse 4
Loaded mouse 5
Loaded mouse 6
Loaded mouse 7
Done with unrew groups
DONE WITH ALL GROUPS AND SITUATIONS -- Ready for PAC analysis


# Save numpy arrrays

In [21]:
print('Rewarded with water arrays (wt & fx)')
print(rew_pac_wt.shape)
print(rew_pac_fx.shape)
print('Rewarded without water arrays (wt & fx)')
print(rew2_pac_wt.shape)
print(rew2_pac_fx.shape)
print('Unrewarded arrays (wt & fx)')
print(unrew_pac_wt.shape)
print(unrew_pac_fx.shape)

Rewarded with water arrays (wt & fx)
(10, 80, 7350)
(8, 80, 7350)
Rewarded without water arrays (wt & fx)
(10, 20, 7350)
(8, 20, 7350)
Unrewarded arrays (wt & fx)
(10, 50, 7350)
(8, 50, 7350)


In [22]:
def save_np_arr(your_name, array):
    arr_save_path = r"D:\mz_Data\saved_dfs\Operant_reward\reward\PAC"
    fn1 = your_name
    
    out1 = arr_save_path + "\\" + fn1
    
    np.save(out1, array)         # saving array
    print('Done!')
    return

In [23]:
save_np_arr('wt_hpc1_rew', rew_pac_wt)
save_np_arr('fx_hpc1_rew', rew_pac_fx)

Done!
Done!


In [24]:
save_np_arr('wt_hpc1_rew2', rew2_pac_wt)
save_np_arr('fx_hpc1_rew2', rew2_pac_fx)

Done!
Done!


In [25]:
save_np_arr('wt_hpc1_unrew', unrew_pac_wt)
save_np_arr('fx_hpc1_unrew', unrew_pac_fx)

Done!
Done!


---

---

---

---

# NOVEL RECORDINGS

In [26]:
start_path_ls=glob.glob(r"G:/Neuropixels/interval_operant_training/operant/raw_data/"+'novel*')
total_rec = 55

end_path = r"\continuous\Neuropix-PXI-100.1\continuous.dat"

In [80]:
alldatals=[]
raw_CC_ls = []
for start_path in start_path_ls:
    ls = []
    CC_num = start_path.split('\\')[-1].split('_')[1]
    et = start_path.split('\\')[-1].split('_')[1] + "_" + start_path.split('\\')[-1].split('_')[2]
    
    for i in range(6, total_rec): # remember, you have to exclude the first short TTL signal trials
        exp_rec_path = rf"\experiment1\recording{i}"
        fileName = start_path + exp_rec_path + end_path
        data = np.memmap(fileName, dtype='int16', mode='c')
        data2 = data.reshape(-1, n_chan)
        ls.append(data2[:samples_tr, 0:384])
        
    alldatals.append(ls)
    raw_CC_ls.append(CC_num)

# alldatals #num_mice x num_trials x num_samples x num_ch
print(len(raw_CC_ls))


16


In [81]:
all_novel_arr = np.load(r"D:\mz_Data\saved_dfs\Operant_reward\reward\lfp_npy\novel_all_rew.npy")

In [82]:
pkl_file = r"D:\mz_Data\saved_dfs\Operant_reward\reward\lfp_npy\novel_cc_ls"

open_file = open(pkl_file, "rb")
CC_ls = pickle.load(open_file)
open_file.close()


In [83]:
print(f'novel: {all_novel_arr.shape} --- mice x ch x samples')

novel: (16, 384, 7350) --- mice x ch x samples


In [84]:
def WTFX_groups(data_array, CC_ls):
    groupA = []
    groupB = []
    for i in range(data_array.shape[0]):
        if (CC_ls[i] == "CC082263") | (CC_ls[i] == "CC067489") | (CC_ls[i] == "CC082260") | (CC_ls[i] == "CC084621"):
            groupA.append(data_array[i])
        elif (CC_ls[i] == "CC082257") | (CC_ls[i] == "CC067431") | (CC_ls[i] == "CC067432") | (CC_ls[i] == "CC082255"):
            groupB.append(data_array[i])
    groupA_arr = np.array(groupA)
    groupB_arr = np.array(groupB)
    return groupA_arr, groupB_arr

def find_strongest_ch(group_arr):
    mean_arr = group_arr.mean(axis=0)
    V1 = mean_arr[200:300, :]
    minim = np.where(V1 == np.amin(V1))
    min_ch = minim[0][0] + 199 # because the first channel is "0" not "1"
    print(min_ch)
    return min_ch# Load previous saved arrays to find strongest channel

In [85]:
# Novel
wt_novel, fx_novel = WTFX_groups(all_novel_arr, CC_ls)

wt_novel_ch = find_strongest_ch(wt_novel)
fx_novel_ch = find_strongest_ch(fx_novel)

264
274


In [86]:
def raw_WTFX_groups(data_ls, raw_CC_ls):
    groupA = []
    groupB = []
    for i in range(len(data_ls)):
        if (raw_CC_ls[i] == "CC082263") | (raw_CC_ls[i] == "CC067489") | (raw_CC_ls[i] == "CC082260") | (raw_CC_ls[i] == "CC084621"):
            groupA.append(data_ls[i])
        elif (raw_CC_ls[i] == "CC082257") | (raw_CC_ls[i] == "CC067431") | (raw_CC_ls[i] == "CC067432") | (raw_CC_ls[i] == "CC082255"):
            groupB.append(data_ls[i])
    groupA_arr = np.array(groupA)
    groupB_arr = np.array(groupB)
    return groupA_arr, groupB_arr

In [89]:
novel_ls = alldatals
raw_wt_novel, raw_fx_novel = raw_WTFX_groups(novel_ls, CC_ls)

In [90]:
times = np.linspace(0, samples_tr/sr, samples_tr)
scale_factor = 0.195
time_arr = [0,0.5,1,1.5,2,2.5]
time_plot = [i*sr for i in time_arr]

In [91]:
# this function creates final CAR filtered arrays for rew, rew2, & unrew for each WT/FX group
# uses "applyCAR" and "notch_filt" functions

def create_reward_arr(reward_list, min_channel):
    rew_ls=[]
    for ii in range(len(reward_list)): #loop through each mouse
        sing_mouse=[]
        for ix in range(len(reward_list[0])): #loop through each trial
            tmp2 = reward_list[ii][ix]
            tmp2 = tmp2.T

            filt_tmp2 = []
            for ch in range(tmp2.shape[0]):
                ch_data = tmp2[ch,:]
                ch_notc_data = mz_LFP.notch_filt(ch_data)
                filt_tmp2.append(ch_notc_data)
            filt_tmp2 = np.array(filt_tmp2)

            CARfilt_tmp2 = mz_LFP.applyCAR(filt_tmp2, pr=0)

            scaled_CARfilt_tmp2 = CARfilt_tmp2[min_channel]*scale_factor

            sing_mouse.append(scaled_CARfilt_tmp2)

        rew_ls.append(sing_mouse)
        print('Loaded mouse {0}'.format(ii))
    rew_arr = np.array(rew_ls)
    
    return rew_arr

In [92]:
#for hpc pac analysis, I am using channel 110 and 140 (maybe?)
wt_novel_ch = 110
fx_novel_ch = 110


In [95]:
# Novel trials
novel_pac_wt = create_reward_arr(raw_wt_novel, wt_novel_ch)
novel_pac_fx = create_reward_arr(raw_fx_novel, fx_novel_ch)
print('Done with novel groups')

Loaded mouse 0
Loaded mouse 1
Loaded mouse 2
Loaded mouse 3
Loaded mouse 4
Loaded mouse 5
Loaded mouse 6
Loaded mouse 7
Loaded mouse 8
Loaded mouse 9
Loaded mouse 0
Loaded mouse 1
Loaded mouse 2
Loaded mouse 3
Loaded mouse 4
Loaded mouse 5
Done with novel groups


In [97]:
def save_np_arr(your_name, array):
    arr_save_path = r"D:\mz_Data\saved_dfs\Operant_reward\reward\PAC"
    fn1 = your_name
    
    out1 = arr_save_path + "\\" + fn1
    
    np.save(out1, array)         # saving array
    print('Done!')
    return

In [98]:
print('Novel arrays (wt & fx) --- mice x trials x samples')
print(novel_pac_wt.shape)
print(novel_pac_fx.shape)

Novel arrays (wt & fx) --- mice x trials x samples
(10, 49, 7350)
(6, 49, 7350)


In [99]:
save_np_arr('wt_hpc1_novel', novel_pac_wt)
save_np_arr('fx_hpc1_novel', novel_pac_fx)

Done!
Done!
